# Echo Chamber Co-Evolution Framework
## Pipeline Completa: Fase 0 → Fase 1 → Fase 2 → Fase 3

Questo notebook esegue la pipeline completa del framework in sequenza.

| Fase | Titolo | Moduli chiave |
|------|--------|---------------|
| **0** | Setup & Baseline | `data_loader`, `extractor` (Forest Fire sampling), `community`, `metrics` |
| **1** | Logica Agente | `agent`, `llm_client`, `state_machine`, `seeder` |
| **2** | Co-evoluzione | `orchestrator`, `gnn`, `rewirer`, `checkpoint` |
| **3** | CELF Fact-Checking | `celf`, `injector`, `influence/metrics` |

> ⚙️ **Configurazione**: modifica `config.yaml` o i parametri nella cella di setup per personalizzare la simulazione.
>
> 🌲 **Sampling**: il sottografo viene estratto con l'algoritmo **Forest Fire** (Leskovec & Faloutsos, KDD 2006) che preserva meglio le proprietà strutturali (distribuzione gradi, clustering, comunità) rispetto a BFS/Random Walk.

## 0. Setup Ambiente

In [1]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

os.environ["GEMINI_API_KEY"] = secret_value_0
os.environ["HF_TOKEN"] = secret_value_1

print("✅ Variabili d'ambiente impostate:")
print(f"   GEMINI_API_KEY = {'*' * 8}{secret_value_0[-4:]}")
print(f"   HF_TOKEN       = {'*' * 8}{secret_value_1[-4:]}")

✅ Variabili d'ambiente impostate:
   GEMINI_API_KEY = ********QdUk
   HF_TOKEN       = ********csqJ


In [ ]:
import sys
import os
from pathlib import Path

# --- Path setup (compatibile locale + Kaggle) ---
if Path('/kaggle/working').exists():
    # Kaggle: clona il repo nella working dir
    PROJECT_ROOT = Path('/kaggle/working/progetto')
    if not PROJECT_ROOT.exists():
        os.system('git clone https://github.com/stefaano19/progettoASM /kaggle/working/progetto')
        os.system('NUMPY_VER=$(python -c "import numpy; print(numpy.__version__)") && pip install -r /kaggle/working/progetto/requirements.txt && pip install --force-reinstall --no-deps numpy==$NUMPY_VER')
else:
    # Locale: la project root è due livelli sopra il notebook
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

In [ ]:
# 4. Infine installa i requirement del tuo progetto locale
# Salva la versione numpy di Kaggle, installa requirements, poi ripristina numpy
import subprocess, sys
try:
    _np_ver = subprocess.check_output([sys.executable, '-c', 'import numpy; print(numpy.__version__)'], text=True).strip()
except Exception:
    _np_ver = None

!pip install -r requirements.txt

if _np_ver:
    !pip install --force-reinstall --no-deps numpy=={_np_ver}
    print(f'✅ numpy ripristinato a {_np_ver}')


## 0.1 Setup vLLM (LLM API Locale)
Installa vLLM, avvia l'API server in background distribuito sulle 2 T4 e scarica il modello Llama 3.

In [4]:
!pip uninstall -y torch torchvision torchaudio torchcodec vllm transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.44.2  
!pip install vllm==0.6.3.post1

import subprocess
cmd = (
    "nohup python -m vllm.entrypoints.openai.api_server "
    "--model unsloth/llama-3-8b-Instruct "
    "--tensor-parallel-size 2 "
    "--gpu-memory-utilization 0.75 "
    "--dtype half "
    "--max-model-len 2048 "
    "> vllm.log 2>&1 &"
)

subprocess.Popen(cmd, shell=True)


Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 104.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.0 MB/s eta 0:00:00:00:

<Popen: returncode: None args: 'nohup python -m vllm.entrypoints.openai.api_...>

In [5]:
import requests, time

# vLLM impiega qualche secondo/minuto ad avviarsi: meglio fare retry
ok = False
for _ in range(120):  # Aumentato a 10 minuti per il download del modello
    try:
        response = requests.get("http://localhost:8000/v1/models", timeout=2)
        if response.status_code == 200:
            ok = True
            break
    except Exception:
        pass
    time.sleep(5)

if ok:
    print(f"✅ vLLM è online! Modelli: {[m['id'] for m in response.json().get('data', [])]}")
else:
    print("❌ Errore: vLLM non sta rispondendo (controlla vllm.log)")
    # utile per debug:
    !tail -n 50 vllm.log

✅ vLLM è online! Modelli: ['unsloth/llama-3-8b-Instruct']


In [6]:
# Parametri globali della pipeline
# Modifica qui invece di editare config.yaml

CONFIG_PATH       = 'config.yaml'
USE_MOCK_LLM      = False       # False = usa il vero LLM
PHASE2_STEPS      = 60          # N. step da eseguire IN QUESTA SESSIONE (non totali!)
PHASE3_STEPS      = 5           # Step post-intervento
CELF_BUDGET_K     = 10          # Fact-checker da iniettare
SKIP_DOWNLOAD     = True        # True se ogbl-collab e' gia' in cache
RESUME_FROM_CKPT  = False       # True per riprendere da checkpoint Fase 2

# --- Sampling (Forest Fire) ---
SAMPLING_STRATEGY = 'forest_fire'  # bfs_seed | random_walk | random_nodes | forest_fire
FOREST_FIRE_PROB  = 0.5            # Forward probability (0.4-0.7); piu' alto = piu' denso
TARGET_NODES      = 5000         # Nodi target nel sottografo

# --- Checkpoint cross-versione Kaggle ---
# Se stai riprendendo da una versione precedente:
#   1. Aggiungi l'output della versione precedente come dataset input
#   2. Imposta PREV_VERSION_CKPT_INPUT al path del file .pkl
#   3. Imposta RESUME_FROM_CKPT = True
PREV_VERSION_CKPT_INPUT = None  # Path del .pkl dalla versione precedente (None = prima run)
PREV_VERSION_CSV_INPUT  = None  # Path del metrics_history.csv dalla versione precedente

print('Parametri pipeline:')
print(f'  CONFIG_PATH             = {CONFIG_PATH}')
print(f'  USE_MOCK_LLM            = {USE_MOCK_LLM}')
print(f'  PHASE2_STEPS            = {PHASE2_STEPS}')
print(f'  PHASE3_STEPS            = {PHASE3_STEPS}')
print(f'  CELF_BUDGET_K           = {CELF_BUDGET_K}')
print(f'  SKIP_DOWNLOAD           = {SKIP_DOWNLOAD}')
print(f'  RESUME_FROM_CKPT        = {RESUME_FROM_CKPT}')
print(f'  SAMPLING_STRATEGY       = {SAMPLING_STRATEGY}')
print(f'  FOREST_FIRE_PROB        = {FOREST_FIRE_PROB}')
print(f'  TARGET_NODES            = {TARGET_NODES}')
print(f'  PREV_VERSION_CKPT_INPUT = {PREV_VERSION_CKPT_INPUT}')
print(f'  PREV_VERSION_CSV_INPUT  = {PREV_VERSION_CSV_INPUT}')

Parametri pipeline:
  CONFIG_PATH             = config.yaml
  USE_MOCK_LLM            = False
  PHASE2_STEPS            = 60
  PHASE3_STEPS            = 5
  CELF_BUDGET_K           = 10
  SKIP_DOWNLOAD           = True
  RESUME_FROM_CKPT        = False
  SAMPLING_STRATEGY       = forest_fire
  FOREST_FIRE_PROB        = 0.5
  TARGET_NODES            = 5000
  PREV_VERSION_CKPT_INPUT = None
  PREV_VERSION_CSV_INPUT  = None


In [7]:
# Carica configurazione e setup globale
from src.utils.config import load_config
from src.utils.seed import set_all_seeds
from src.utils.logger import setup_logging
import logging

cfg = load_config(CONFIG_PATH)

# --- Override dal notebook ---
cfg.llm.backend = "local"
cfg.llm.local.model = "unsloth/llama-3-8b-Instruct"
cfg.subgraph.target_nodes = TARGET_NODES
cfg.subgraph.strategy = SAMPLING_STRATEGY
cfg.subgraph.forward_prob = FOREST_FIRE_PROB
# -----------------------
setup_logging('INFO')
set_all_seeds(cfg.execution.random_seed)

print(f'Config caricata: hash={cfg.config_hash}')
print(f'Random seed: {cfg.execution.random_seed}')
print(f'Embedding dim: {cfg.gnn.embedding_dim}')
print(f'Max steps: {cfg.simulation.max_steps}')
print(f'Sampling: strategy={cfg.subgraph.strategy}, forward_prob={cfg.subgraph.forward_prob}')
print(f'Max concurrent LLM: {cfg.llm.max_concurrent_requests}')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

---
## Fase 0 — Setup & Baseline

Carica `ogbl-collab`, estrae il sottografo, rileva community, calcola metriche baseline.

In [ ]:
import numpy as np
import networkx as nx

# --- 1. Caricamento dati ---
from src.graph.data_loader import load_collab_graph, graph_summary

print('⏳ Caricamento ogbl-collab...')
G_full, node_features = load_collab_graph(cfg)
summary = graph_summary(G_full)
print(f'✅ Grafo completo: {summary["num_nodes"]:,} nodi | {summary["num_edges"]:,} archi')

In [ ]:
# --- 2. Estrazione sottografo ---
from src.graph.extractor import extract_subgraph

print(f'⏳ Estrazione sottografo (target={cfg.subgraph.target_nodes} nodi)...')
print(f'   Strategia: {cfg.subgraph.strategy}')
if cfg.subgraph.strategy == 'forest_fire':
    print(f'   Forward prob: {cfg.subgraph.forward_prob}')
subG, sub_features, node_map = extract_subgraph(G_full, node_features, cfg)
print(f'✅ Sottografo: {subG.number_of_nodes()} nodi | {subG.number_of_edges()} archi')
print(f'   Densità: {nx.density(subG):.4f}')
print(f'   Rapporto: {subG.number_of_nodes()/G_full.number_of_nodes()*100:.1f}% dei nodi originali')

### 📐 Validazione Strutturale: Sottografo vs Grafo Originale

Confronto delle proprietà strutturali per verificare che il campionamento (Forest Fire)
preservi le caratteristiche fondamentali del grafo originale.

In [ ]:
# --- Validazione strutturale: sottografo vs originale ---
import time

def _quick_metrics(G, label, sample_size=3000):
    """Calcola metriche strutturali chiave per un grafo."""
    t0 = time.time()
    n = G.number_of_nodes()
    degrees = np.array([d for _, d in G.degree()])

    # Clustering (campionato per grafi grandi)
    if n > 10_000:
        sample_nodes = list(np.random.default_rng(42).choice(
            list(G.nodes()), size=min(sample_size, n), replace=False))
        cc = nx.average_clustering(G, nodes=sample_nodes)
    else:
        cc = nx.average_clustering(G)

    # Modularity (label propagation — veloce)
    comms = list(nx.community.label_propagation_communities(G))
    mod_q = nx.community.modularity(G, comms)

    elapsed = time.time() - t0
    return {
        'label': label, 'nodes': n, 'edges': G.number_of_edges(),
        'density': nx.density(G), 'avg_degree': float(degrees.mean()),
        'max_degree': int(degrees.max()), 'std_degree': float(degrees.std()),
        'clustering': cc, 'modularity': mod_q, 'n_communities': len(comms),
        'degree_seq': sorted(degrees, reverse=True), 'time': elapsed,
    }

print('⏳ Calcolo metriche grafo originale (LCC)...')
# Prendi la LCC del grafo completo (gia' in memoria da Cell 13)
lcc_nodes = max(nx.connected_components(G_full), key=len)
G_lcc = G_full.subgraph(lcc_nodes)
m_full = _quick_metrics(G_lcc, 'Originale (LCC)')

print('⏳ Calcolo metriche sottografo...')
m_sub = _quick_metrics(subG, f'Sottografo ({cfg.subgraph.strategy})')

# --- Tabella di confronto ---
print('\n' + '=' * 72)
print('VALIDAZIONE STRUTTURALE — Sottografo vs Originale')
print('=' * 72)
print(f'{"Metrica":<22} {"Originale":>14} {"Sottografo":>14} {"Δ relativo":>14}')
print('-' * 72)

comparisons = [
    ('Nodi',            'nodes',         '%d'),
    ('Archi',           'edges',         '%d'),
    ('Densità',         'density',       '%.6f'),
    ('Grado medio ⟨k⟩', 'avg_degree',   '%.1f'),
    ('Grado max',       'max_degree',    '%d'),
    ('σ(grado)',        'std_degree',    '%.1f'),
    ('Clustering CC',   'clustering',    '%.4f'),
    ('Modularity Q',    'modularity',    '%.4f'),
    ('N. comunità',     'n_communities', '%d'),
]

for label, key, fmt in comparisons:
    v_full = m_full[key]
    v_sub  = m_sub[key]
    if v_full != 0:
        delta_pct = (v_sub - v_full) / abs(v_full) * 100
        delta_s = f'{delta_pct:+.1f}%'
    else:
        delta_s = 'n/a'
    print(f'{label:<22} {fmt % v_full:>14} {fmt % v_sub:>14} {delta_s:>14}')

print('=' * 72)
ratio = m_sub['nodes'] / m_full['nodes'] * 100
print(f'Rapporto campionamento: {ratio:.1f}% dei nodi originali')
print(f'Tempo calcolo: originale={m_full["time"]:.1f}s, sottografo={m_sub["time"]:.1f}s')

# --- Plot: CCDF dei gradi ---
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: Degree CCDF (log-log)
    ax = axes[0]
    for m, color, ls in [(m_full, '#2196F3', '-'), (m_sub, '#FF5722', '--')]:
        degs = np.array(m['degree_seq'])
        unique_d = np.sort(np.unique(degs))
        ccdf = np.array([np.mean(degs >= d) for d in unique_d])
        ax.loglog(unique_d, ccdf, ls, color=color, label=m['label'], linewidth=1.8, alpha=0.85)
    ax.set_xlabel('Grado (k)', fontsize=11)
    ax.set_ylabel('P(X ≥ k) — CCDF', fontsize=11)
    ax.set_title('Distribuzione dei Gradi', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which='both')

    # Panel 2: Bar chart metriche normalizzate
    ax = axes[1]
    metric_labels = ['CC', 'Mod Q', '⟨k⟩ (norm)', 'Densità (norm)']
    full_vals = [m_full['clustering'], m_full['modularity'], 1.0, 1.0]
    sub_vals = [
        m_sub['clustering'],
        m_sub['modularity'],
        m_sub['avg_degree'] / max(m_full['avg_degree'], 1e-8),
        m_sub['density'] / max(m_full['density'], 1e-8),
    ]
    x = np.arange(len(metric_labels))
    w = 0.35
    ax.bar(x - w/2, full_vals, w, label='Originale', color='#2196F3', alpha=0.8)
    ax.bar(x + w/2, sub_vals, w, label='Sottografo', color='#FF5722', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.set_ylabel('Valore', fontsize=11)
    ax.set_title('Confronto Metriche Strutturali', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    fig_path = cfg.project_root / cfg.paths.figures / 'subgraph_validation.png'
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'📊 Figura salvata: {fig_path}')
except Exception as e:
    print(f'⚠️ Plot non disponibile: {e}')

In [ ]:
# --- 3. Community detection ---
from src.graph.community import detect_communities, community_stats

print(f'⏳ Community detection (algoritmo={cfg.community.algorithm})...')
community_map, n_comm, q = detect_communities(subG, cfg)
stats = community_stats(subG, community_map)
print(f'✅ Community: {n_comm} trovate | Q={q:.4f}')
print(f'   Dimensioni: min={stats["min_size"]} max={stats["max_size"]} media={stats["mean_size"]:.1f}')

In [ ]:
# --- 4. Metriche baseline ---
from src.graph.metrics import compute_all_metrics, compute_centralities

print('⏳ Calcolo metriche baseline...')
baseline_metrics = compute_all_metrics(subG, cfg, community_map=community_map)
centralities = compute_centralities(subG, cfg)

print('\n📊 METRICHE BASELINE (Fase 0)')
print('=' * 40)
for k, v in baseline_metrics.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.4f}')
    elif isinstance(v, int):
        print(f'  {k:30s}: {v:,}')

# Top-5 PageRank
top_pr = sorted(
    [(n, d.get('pagerank', 0)) for n, d in centralities.items()],
    key=lambda x: x[1], reverse=True
)[:5]
print(f'\n  Top-5 PageRank: {[(n, f"{pr:.4f}") for n, pr in top_pr]}')

---
## Fase 1+2 — Agenti LLM e Co-evoluzione

L'**Orchestratore** gestisce l'intera pipeline:
1. Inizializza embeddings, NetworkManager, agenti LLM e pazienti zero
2. Esegue il loop co-evolutivo: **Agenti → GNN fine-tuning → Rewiring**
3. Salva checkpoint e metriche CSV ad ogni step

> 🧠 **LLM reale**: gli agenti usano il server vLLM locale (Llama 3 8B) per generare opinioni e transizioni di stato.
> Non viene usato nessun mock.

In [ ]:
# ============================================================
# IMPORT CHECKPOINT DA VERSIONE PRECEDENTE (cross-session Kaggle)
# ============================================================
# Se PREV_VERSION_CKPT_INPUT e' impostato, copia il checkpoint
# dall'output della versione precedente nella directory checkpoints
# del progetto, cosi' RESUME_FROM_CKPT=True trovera' il file.
import shutil
from pathlib import Path

if RESUME_FROM_CKPT and PREV_VERSION_CKPT_INPUT is not None:
    src_ckpt = Path(PREV_VERSION_CKPT_INPUT)
    if not src_ckpt.exists():
        raise FileNotFoundError(f'Checkpoint non trovato: {src_ckpt}')
    ckpt_dir = PROJECT_ROOT / 'results' / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    dst = ckpt_dir / src_ckpt.name
    shutil.copy2(src_ckpt, dst)
    print(f'\u2705 Checkpoint importato: {src_ckpt.name} \u2192 {dst}')
elif RESUME_FROM_CKPT:
    # Controlla se esiste gia' un checkpoint locale
    ckpt_dir = PROJECT_ROOT / 'results' / 'checkpoints'
    existing = sorted(ckpt_dir.glob('ckpt_step_*.pkl')) if ckpt_dir.exists() else []
    if existing:
        print(f'\u2705 Checkpoint locale trovato: {existing[-1].name}')
    else:
        print('\u26a0\ufe0f  RESUME_FROM_CKPT=True ma nessun checkpoint trovato.')
        print('   Imposta PREV_VERSION_CKPT_INPUT per importare da versione precedente.')
        RESUME_FROM_CKPT = False  # fallback: nuova run
else:
    print('\u2139\ufe0f  Prima sessione, parto da zero.')

In [ ]:
# ============================================================
# IMPORT CSV METRICHE DA VERSIONE PRECEDENTE
# ============================================================
# Se PREV_VERSION_CSV_INPUT e' impostato, copia il CSV dalla
# versione precedente nella directory results/ del progetto.
# L'orchestratore aprira' il CSV in append mode e accodera' i nuovi step,
# mantenendo la storia completa di tutte le sessioni.
import shutil
from pathlib import Path

results_dir = PROJECT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
csv_dst = results_dir / 'metrics_history.csv'

if RESUME_FROM_CKPT and PREV_VERSION_CSV_INPUT is not None:
    src_csv = Path(PREV_VERSION_CSV_INPUT)
    if src_csv.exists():
        shutil.copy2(src_csv, csv_dst)
        import pandas as pd
        df_prev = pd.read_csv(csv_dst)
        print(f'\u2705 CSV metriche importato: {src_csv.name}')
        print(f'   Righe precedenti: {len(df_prev)} step (step {df_prev["step"].min()}..{df_prev["step"].max()})')
    else:
        print(f'\u26a0\ufe0f  CSV non trovato: {src_csv} \u2014 si parte da zero.')
elif RESUME_FROM_CKPT and csv_dst.exists():
    import pandas as pd
    df_prev = pd.read_csv(csv_dst)
    print(f'\u2705 CSV locale trovato: {len(df_prev)} step precedenti')
else:
    print('\u2139\ufe0f  Nessun CSV precedente da importare (prima sessione).')

In [ ]:
import uuid
from src.orchestrator import SimulationOrchestrator

print('⏳ Costruzione SimulationOrchestrator...')
orch = SimulationOrchestrator.build_from_config(
    cfg,
    use_mock_llm=USE_MOCK_LLM,
    resume=RESUME_FROM_CKPT,
)
print(f'✅ Orchestratore pronto')
print(f'   Nodi: {orch.network_manager.num_nodes}')
print(f'   Archi: {orch.network_manager.num_edges}')

In [ ]:
# Tracking metriche per step (per i grafici finali)
metrics_history = []

# --- Resume: calcola il range corretto ---
# orch.resume_step = 0 se nuova run, = ckpt_step+1 se resume
start_step = orch.resume_step
end_step   = start_step + PHASE2_STEPS

print(f'\U0001f680 Avvio loop co-evolutivo: step {start_step} \u2192 {end_step - 1} ({PHASE2_STEPS} step questa sessione)...')
print(f'   (PHASE2_STEPS conta gli step di QUESTA sessione, non il totale)')
print('=' * 65)
print(f'{"Step":>5}  {"S":>6}  {"I":>6}  {"R":>6}  {"F":>6}  {"ECI":>8}  {"Loss":>8}  {"Edges":>7}')
print('-' * 65)

for t in range(start_step, end_step):
    step_metrics = orch._run_step(t)
    s_counts = orch.state_summary
    metrics_history.append({'step': t, **step_metrics, **s_counts})

    print(
        f'{t:>5}  {s_counts.get("S",0):>6}  {s_counts.get("I",0):>6}  '
        f'{s_counts.get("R",0):>6}  {s_counts.get("F",0):>6}  '
        f'{step_metrics.get("echo_chamber_index") or 0.0:>8.4f}  '
        f'{step_metrics.get("gnn_loss") or 0.0:>8.4f}  '
        f'{step_metrics.get("num_edges") or 0:>7}'
    )

print('=' * 65)
print(f'\u2705 Loop completato. Step eseguiti: {start_step}\u2013{end_step - 1}')
print(f'   Ultimo checkpoint automatico: ckpt_step_{orch.current_step:04d}.pkl')

In [ ]:
# Chiudi il file CSV delle metriche in modo pulito
orch.close()
print('\u2705 Orchestratore chiuso (CSV metriche flushed).')

In [ ]:
# Visualizzazione evoluzione nel tempo
try:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    steps = [m['step'] for m in metrics_history]
    n_S = [m.get('S', 0) for m in metrics_history]
    n_I = [m.get('I', 0) for m in metrics_history]
    n_R = [m.get('R', 0) for m in metrics_history]
    n_F = [m.get('F', 0) for m in metrics_history]
    eci = [m.get('echo_chamber_index') or 0.0 for m in metrics_history]
    loss= [m.get('gnn_loss') or 0.0 for m in metrics_history]
    edges=[m.get('num_edges') or 0 for m in metrics_history]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Echo Chamber Co-Evolution — Fase 2', fontsize=13, fontweight='bold')
    
    # Plot 1: Dinamica stati
    axes[0].stackplot(steps, n_S, n_I, n_R, n_F,
                      labels=['S (Susceptible)', 'I (Infected)', 'R (Resistant)', 'F (Fact-Checker)'],
                      colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.8)
    axes[0].set_title('Dinamica degli Stati')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('N. Nodi')
    axes[0].legend(loc='upper left', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Echo Chamber Index
    axes[1].plot(steps, eci, color='#8e44ad', linewidth=2, marker='o', markersize=4)
    axes[1].fill_between(steps, eci, alpha=0.2, color='#8e44ad')
    axes[1].set_title('Echo Chamber Index (ECI)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('ECI')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 1)
    
    # Plot 3: Archi nel tempo (rewiring)
    axes[2].plot(steps, edges, color='#16a085', linewidth=2, marker='s', markersize=4)
    axes[2].fill_between(steps, edges, alpha=0.2, color='#16a085')
    axes[2].set_title('Evoluzione Archi (Rewiring)')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('N. Archi')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = cfg.project_root / cfg.paths.figures / 'phase2_evolution.png'
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'📊 Grafico salvato: {fig_path}')
    plt.show()

except ImportError:
    print('matplotlib non disponibile — skip grafici')

---
## Fase 3 — CELF Fact-Checking

Selezione ottimale dei seed fact-checker tramite **CELF** (Cost-Effective Lazy Forward), iniezione e misurazione dell'impatto.

**Flusso:**
1. Snapshot metriche pre-intervento (baseline)
2. CELF seleziona i `k` nodi ottimali per massimizzare lo spread fact-checking
3. Iniezione: i nodi selezionati diventano Fact-Checker (stato F)
4. N step post-intervento per osservare la propagazione
5. Confronto before/after con delta metriche

> 📊 Il CSV include una colonna `phase` per distinguere step Fase 2 da Fase 3.

In [ ]:
from src.graph.metrics import compute_all_metrics
from src.agents.state_machine import StateMachine

# Snapshot pre-intervento
nm_ph3 = orch.network_manager
community_map_ph3 = nm_ph3._community_map

pre_belief_map = nm_ph3.get_belief_map()
pre_metrics = compute_all_metrics(nm_ph3.G, cfg, community_map_ph3, pre_belief_map)
pre_states = nm_ph3.get_all_states()
pre_counts = StateMachine.count_states(pre_states)
n_total = max(nm_ph3.num_nodes, 1)
pre_metrics['infection_rate'] = pre_counts['I'] / n_total
pre_metrics['n_S'] = pre_counts['S']
pre_metrics['n_I'] = pre_counts['I']
pre_metrics['n_R'] = pre_counts['R']
pre_metrics['n_F'] = pre_counts['F']

print('📊 BASELINE PRE-INTERVENTO')
print(f'  S={pre_counts["S"]}  I={pre_counts["I"]}  R={pre_counts["R"]}  F={pre_counts["F"]}')
print(f'  Infection Rate : {pre_metrics["infection_rate"]:.4f}')
print(f'  ECI            : {pre_metrics.get("echo_chamber_index") or 0.0:.4f}')
print(f'  Modularity Q   : {pre_metrics.get("modularity_q") or 0.0:.4f}')

In [ ]:
# --- CELF: selezione seed ---
from src.influence.celf import CELF

print(f'⏳ CELF: selezione {CELF_BUDGET_K} seed fact-checker...')
celf = CELF(cfg)
celf_seeds = celf.select(
    G=nm_ph3.G,
    budget_k=CELF_BUDGET_K,
    agent_states=nm_ph3.get_all_states(),
)
print(f'✅ CELF seeds: {celf_seeds}')

# Stima spread
estimated_spread = celf.estimate_spread(nm_ph3.G, celf_seeds, nm_ph3.get_all_states())
print(f'   Spread stimato: {estimated_spread:.1f} nodi ({estimated_spread/n_total*100:.1f}%)')

In [ ]:
# --- Iniezione fact-checker ---
from src.influence.injector import FactCheckerInjector

injector = FactCheckerInjector(cfg)
injected = injector.inject(nm_ph3, celf_seeds, step=orch.current_step)
print(f'✅ Fact-checker iniettati: {len(injected)} nodi → {injected}')

# Aggiorna stato agenti in memoria
from src.agents.agent import AgentState
for node_id in injected:
    if node_id in orch._agents:
        orch._agents[node_id]._state = AgentState.from_str('F')

In [ ]:
# --- Step post-intervento ---
print(f'⏳ Esecuzione {PHASE3_STEPS} step post-intervento...')
post_history = []

# Marca i prossimi step come Fase 3 nel CSV
orch._phase = "3"

start_step = orch.current_step + 1
for t in range(start_step, start_step + PHASE3_STEPS):
    step_metrics = orch._run_step(t)
    s_counts = orch.state_summary
    post_history.append({'step': t, **step_metrics, **s_counts})
    print(
        f'  Step {t}: S={s_counts.get("S",0)} I={s_counts.get("I",0)} '
        f'R={s_counts.get("R",0)} F={s_counts.get("F",0)} '
        f'ECI={step_metrics.get("echo_chamber_index") or 0.0:.4f}'
    )

print('✅ Step post-intervento completati')

In [ ]:
# --- Report completo ---
from src.influence.metrics import compute_full_influence_report

final_report = compute_full_influence_report(
    G=nm_ph3.G,
    agent_states=nm_ph3.get_all_states(),
    community_map=community_map_ph3,
    baseline_metrics=pre_metrics,
    cfg=cfg,
)
final_report['celf_seeds'] = celf_seeds
final_report['injected_nodes'] = list(injected)
final_report['n_post_steps'] = PHASE3_STEPS
final_report['sampling_strategy'] = cfg.subgraph.strategy
final_report['forward_prob'] = cfg.subgraph.forward_prob

print('\n' + '=' * 65)
print('CONFRONTO BEFORE / AFTER INTERVENTO CELF')
print('=' * 65)
print(f'{"Metrica":<25}  {"PRIMA":>10}  {"DOPO":>10}  {"DELTA":>10}')
print('-' * 65)

metrics_to_compare = [
    ('Infection Rate',      'infection_rate',       '%.4f'),
    ('Echo Chamber Idx',    'echo_chamber_index',   '%.4f'),
    ('Modularity Q',        'modularity_q',         '%.4f'),
    ('Belief Polarisation', 'belief_polarisation',  '%.4f'),
    ('Nodi S',              'n_S',                  '%d'),
    ('Nodi I',              'n_I',                  '%d'),
    ('Nodi R',              'n_R',                  '%d'),
    ('Nodi F',              'n_F',                  '%d'),
]

for label, key, fmt in metrics_to_compare:
    before = pre_metrics.get(key)
    after  = final_report.get(key)
    delta  = final_report.get(f'delta_{key}')
    before_s = (fmt % before) if before is not None else 'n/a'
    after_s  = (fmt % after)  if after  is not None else 'n/a'
    delta_s  = (f'{delta:+.4f}' if isinstance(delta, float) else
                f'{delta:+d}' if isinstance(delta, int) else 'n/a')
    print(f'{label:<25}  {before_s:>10}  {after_s:>10}  {delta_s:>10}')

print('-' * 65)
print(f'{"Fact-Checker Spread":<25}  {"":>10}  {final_report.get("fcs", 0.0):>10.4f}')
print(f'{"Avg Reach per FC":<25}  {"":>10}  {final_report.get("avg_reach_per_fc", 0.0):>10.1f}')
print('=' * 65)

# --- Salva report JSON ---
import json
report_path = cfg.project_root / cfg.paths.results / 'phase3_report.json'
serializable = {k: v for k, v in final_report.items()
                if isinstance(v, (int, float, str, list, dict, bool, type(None)))}
with open(report_path, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'\n💾 Report salvato: {report_path}')

In [ ]:
# --- Visualizzazione Phase 3 ---
try:
    import matplotlib.pyplot as plt
    
    # Combina metriche storia completa
    all_history = metrics_history + post_history
    all_steps = [m['step'] for m in all_history]
    all_n_I = [m.get('I', m.get('n_I', 0)) for m in all_history]
    all_n_F = [m.get('F', m.get('n_F', 0)) for m in all_history]
    all_eci = [m.get('echo_chamber_index') or 0.0 for m in all_history]
    
    injection_step = start_step - 1
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Echo Chamber Framework — Effetto Intervento CELF', fontsize=13, fontweight='bold')
    
    # Plot: Nodi I e F nel tempo
    ax1.plot(all_steps, all_n_I, color='#e74c3c', linewidth=2, label='I (Infected)', marker='o', ms=4)
    ax1.plot(all_steps, all_n_F, color='#f39c12', linewidth=2, label='F (Fact-Checker)', marker='s', ms=4)
    ax1.axvline(x=injection_step, color='black', linestyle='--', alpha=0.7, label=f'Iniezione FC (step {injection_step})')
    ax1.set_title('Diffusione: Infetti vs Fact-Checker')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('N. Nodi')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Plot: ECI nel tempo
    ax2.plot(all_steps, all_eci, color='#8e44ad', linewidth=2, marker='o', ms=4)
    ax2.axvline(x=injection_step, color='black', linestyle='--', alpha=0.7, label=f'Iniezione FC')
    ax2.fill_between(all_steps, all_eci, alpha=0.15, color='#8e44ad')
    ax2.set_title('Echo Chamber Index nel Tempo')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('ECI')
    ax2.set_ylim(0, 1)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = cfg.project_root / cfg.paths.figures / 'phase3_intervention.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'📊 Grafico salvato: {fig_path}')
    plt.show()

except ImportError:
    print('matplotlib non disponibile — skip grafici')

---
## Riepilogo Pipeline

In [ ]:
import json

pipeline_summary = {
    'config_hash': cfg.config_hash,
    'seed': cfg.execution.random_seed,
    'phase0': {
        'n_nodes': subG.number_of_nodes(),
        'n_edges': subG.number_of_edges(),
        'n_communities': n_comm,
        'modularity_q_baseline': round(q, 4),
        'echo_chamber_index_baseline': round(baseline_metrics.get('echo_chamber_index') or 0.0, 4),
    },
    'phase1': {
        'n_patient_zeros': len(patient_zero_ids),
        'patient_zero_ids': patient_zero_ids,
        'strategy': cfg.simulation.seeder_strategy,
    },
    'phase2': {
        'n_steps': PHASE2_STEPS,
        'llm_mode': 'mock' if USE_MOCK_LLM else 'api',
        'final_state_counts': pre_counts,
        'infection_rate_post_phase2': round(pre_metrics['infection_rate'], 4),
        'eci_post_phase2': round(pre_metrics.get('echo_chamber_index') or 0.0, 4),
    },
    'phase3': {
        'celf_seeds': celf_seeds,
        'injected_nodes': injected,
        'budget_k': CELF_BUDGET_K,
        'n_post_steps': PHASE3_STEPS,
        'fcs': round(final_report.get('fcs', 0.0), 4),
        'delta_infection_rate': round(final_report.get('delta_infection_rate', 0.0), 4),
        'delta_eci': round(final_report.get('delta_echo_chamber_index', 0.0), 4),
        'final_n_F': final_report.get('n_F', 0),
    },
}

summary_path = cfg.project_root / cfg.paths.results / 'pipeline_summary.json'
with open(summary_path, 'w') as f:
    json.dump(pipeline_summary, f, indent=2)

print('\n' + '=' * 60)
print('PIPELINE COMPLETATA ✅')
print('=' * 60)
print(f'  Nodi simulati    : {subG.number_of_nodes()}')
print(f'  Archi iniziali   : {subG.number_of_edges()}')
print(f'  Community        : {n_comm} (Q baseline={q:.4f})')
print(f'  Step Fase 2      : {PHASE2_STEPS}')
print(f'  Infection rate   : {pre_metrics["infection_rate"]:.4f} → {final_report.get("infection_rate", 0.0):.4f}')
print(f'  ECI              : {pre_metrics.get("echo_chamber_index") or 0.0:.4f} → {final_report.get("echo_chamber_index") or 0.0:.4f}')
print(f'  CELF seeds       : {len(celf_seeds)} nodi')
print(f'  FCS post-int.    : {final_report.get("fcs", 0.0):.4f}')
print('=' * 60)
print(f'Riepilogo salvato: {summary_path}')

## Visualizzazione Finale: Come è cambiata la Rete
Esegue un confronto visuale tra la struttura della rete all'inizio della simulazione e la struttura alla fine dell'ultimo step, campionando i nodi principali per mantenere il grafico leggibile.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pickle
import glob
import os

# ---------------------------------------------------------
# VISUALIZZAZIONE RIASSUNTIVA: RETE INIZIALE VS RETE FINALE
# ---------------------------------------------------------
print("Generazione del grafico riassuntivo in corso...")

# 1. Carica il grafo iniziale
init_path = 'data/processed/subgraph.gpickle'
if os.path.exists(init_path):
    with open(init_path, 'rb') as f:
        data = pickle.load(f)
    G_init = data['graph'] if isinstance(data, dict) else data
else:
    print("Grafo iniziale non trovato.")
    G_init = None

# 2. Carica l'ultimo checkpoint per la rete finale
checkpoints = sorted(glob.glob('results/checkpoints/ckpt_step_*.pkl'))
if checkpoints and G_init:
    last_ckpt = checkpoints[-1]
    with open(last_ckpt, 'rb') as f:
        ckpt_data = pickle.load(f)
    
    # Estrai il grafo finale dal NetworkManager salvato
    G_final = ckpt_data.network_manager.G

    # Campionamento: visualizzare 30k nodi creerebbe una macchia nera.
    # Selezioniamo un campione dei 150 nodi più connessi (hub)
    degrees = dict(G_init.degree())
    top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:150]
    
    # Estrai i sottografi per il campione
    sub_init = G_init.subgraph(top_nodes)
    sub_final = G_final.subgraph(top_nodes)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Fissa il layout in base al grafo iniziale per poter confrontare visivamente
    pos = nx.spring_layout(sub_init, seed=42)
    
    # Disegna Rete Iniziale
    nx.draw(sub_init, pos, ax=ax1, node_size=30, node_color='#3498db', edge_color='gray', alpha=0.7)
    ax1.set_title(f"Rete Iniziale (Campione Hub: {sub_init.number_of_nodes()} nodi, {sub_init.number_of_edges()} archi)", fontsize=14, pad=15)
    
    # Disegna Rete Finale
    nx.draw(sub_final, pos, ax=ax2, node_size=30, node_color='#e74c3c', edge_color='gray', alpha=0.7)
    ax2.set_title(f"Rete Finale (Stesso Campione: {sub_final.number_of_nodes()} nodi, {sub_final.number_of_edges()} archi)", fontsize=14, pad=15)
    
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/network_comparison_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Impossibile caricare i dati per il confronto. Verifica che la simulazione abbia salvato i checkpoint.")
